In [1]:
# ==========================================================
# ACTIVIDAD 21. Comparación con Gaussian Mixture (3 componentes)
# Mismos datos y misma regla: se ajusta SOLO con empresas y bloque
# de entrenamiento y se aplica sin reentrenar a todas las filas.
# ¿Encuentra los mismos regímenes que K-Means?
# ==========================================================
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture

SEMILLA = 42
K = 3
UMBRAL_DUDOSO = 0.6                # probabilidad máxima por debajo de la cual la asignación es dudosa
MUESTRA_SILUETA = 20_000
VARIABLES = ["volatilidad_21", "cambio_volatilidad", "rendimiento_21", "caida_maximo_63", "volumen_anormal"]
COLUMNAS_Z = ["z_" + v for v in VARIABLES]

drive.mount("/content/drive")
RAIZ = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ----------------------------------------------------------
# 1. Datos y régimen de K-Means (modelo definitivo, sin cambios)
# ----------------------------------------------------------
df = pd.read_parquet(RAIZ / "01_datos" / "variables_preparadas.parquet")
for col in ["ticker", "grupo", "bloque"]:
    df[col] = df[col].astype(str)
X = df[COLUMNAS_Z].to_numpy()
kmeans = joblib.load(RAIZ / "02_modelos" / "kmeans_k3.joblib")
nombres = kmeans["nombres"]
df["kmeans"] = pd.Series(kmeans["modelo"].predict(X), index=df.index).map(kmeans["renumerar"])

es_ajuste = ((df["grupo"] == "entrenamiento") & (df["bloque"] == "entrenamiento")).to_numpy()
df["celda"] = df["grupo"] + " / " + df["bloque"]
ORDEN_CELDAS = ["entrenamiento / entrenamiento", "entrenamiento / validacion", "entrenamiento / prueba",
                "reservada / entrenamiento", "reservada / validacion", "reservada / prueba"]

# ----------------------------------------------------------
# 2. Ajustar Gaussian Mixture SOLO con el subconjunto de ajuste
# ----------------------------------------------------------
inicio = time.time()
gmm = GaussianMixture(n_components=K, covariance_type="full", n_init=3, max_iter=300,
                      random_state=SEMILLA).fit(X[es_ajuste])
print(f"Gaussian Mixture ajustado en {time.time() - inicio:.0f} s | convergió: {gmm.converged_}")

# Ordenar los componentes por volatilidad media (1 = menor), igual que en K-Means
orden = np.argsort(gmm.means_[:, 0])
renumerar = {int(antiguo): nuevo + 1 for nuevo, antiguo in enumerate(orden)}
probabilidades = gmm.predict_proba(X)[:, orden]               # columnas: régimen 1, 2, 3
df["gmm"] = probabilidades.argmax(axis=1) + 1
df["prob_max"] = probabilidades.max(axis=1)

# ----------------------------------------------------------
# 3. ¿Coinciden K-Means y Gaussian Mixture?
# ----------------------------------------------------------
ent = df[es_ajuste]
print("\n=== CRUCE EN EL SUBCONJUNTO DE AJUSTE (% del total de filas) ===")
cruce = pd.crosstab(ent["kmeans"].map(nombres), ent["gmm"].map(nombres), normalize="all") * 100
print(cruce.reindex(index=list(nombres.values()), columns=list(nombres.values())))
print("(filas: K-Means | columnas: Gaussian Mixture; la diagonal es el acuerdo)")

filas = []
for celda in ORDEN_CELDAS:
    d = df[df["celda"] == celda]
    filas.append({"celda": celda,
                  "acuerdo_pct": (d["kmeans"] == d["gmm"]).mean() * 100,
                  "ari_kmeans_vs_gmm": adjusted_rand_score(d["kmeans"], d["gmm"]),
                  **{f"gmm_{nombres[r]}_pct": (d["gmm"] == r).mean() * 100 for r in nombres},
                  "prob_max_media": d["prob_max"].mean(),
                  "dudosas_pct": (d["prob_max"] < UMBRAL_DUDOSO).mean() * 100})
comparacion = pd.DataFrame(filas).set_index("celda")
print("\n=== ACUERDO, PROPORCIONES DE GAUSSIAN MIXTURE Y CONFIANZA POR CELDA ===")
print(comparacion)

# ----------------------------------------------------------
# 4. Perfiles de Gaussian Mixture frente a K-Means (medianas, subconjunto de ajuste)
# ----------------------------------------------------------
perfil_gmm = ent.groupby("gmm")[VARIABLES].median().rename(index=nombres)
perfil_km = ent.groupby("kmeans")[VARIABLES].median().rename(index=nombres)
print("\n=== PERFIL GAUSSIAN MIXTURE (medianas) ===")
print(perfil_gmm)
print("\n=== PERFIL K-MEANS (medianas, para comparar) ===")
print(perfil_km)

# ----------------------------------------------------------
# 5. Silueta de ambos modelos sobre las mismas muestras
# ----------------------------------------------------------
print("\n=== SILUETA (misma muestra de 20.000 filas) ===")
for celda in ["entrenamiento / entrenamiento", "reservada / prueba"]:
    d = df[df["celda"] == celda]
    m = d.sample(min(MUESTRA_SILUETA, len(d)), random_state=SEMILLA)
    print(f"{celda:32s} K-Means {silhouette_score(m[COLUMNAS_Z], m['kmeans']):.3f} | "
          f"Gaussian Mixture {silhouette_score(m[COLUMNAS_Z], m['gmm']):.3f}")

# ----------------------------------------------------------
# 6. Guardar modelo, comparación y probabilidades (para el dashboard)
# ----------------------------------------------------------
joblib.dump({"modelo": gmm, "renumerar": renumerar, "nombres": nombres},
            RAIZ / "02_modelos" / "gmm_k3.joblib")
comparacion.to_csv(RAIZ / "02_modelos" / "comparacion_kmeans_gmm.csv")
salida = df[["ticker", "date", "kmeans", "gmm"]].copy()
for r, nombre in nombres.items():
    salida[f"prob_{nombre.lower().replace('é', 'e')}"] = probabilidades[:, r - 1].astype("float32")
salida["ticker"] = salida["ticker"].astype("category")
salida.to_parquet(RAIZ / "01_datos" / "regimenes_gmm.parquet", index=False)
print("\nGuardado: 02_modelos/gmm_k3.joblib, comparacion_kmeans_gmm.csv y 01_datos/regimenes_gmm.parquet")

Mounted at /content/drive
Gaussian Mixture ajustado en 73 s | convergió: True

=== CRUCE EN EL SUBCONJUNTO DE AJUSTE (% del total de filas) ===
gmm      Calma  Impulso  Estrés
kmeans                         
Calma   31.223   19.662   2.024
Impulso 13.645   15.159   4.117
Estrés   0.000    4.203   9.967
(filas: K-Means | columnas: Gaussian Mixture; la diagonal es el acuerdo)

=== ACUERDO, PROPORCIONES DE GAUSSIAN MIXTURE Y CONFIANZA POR CELDA ===
                               acuerdo_pct  ari_kmeans_vs_gmm  gmm_Calma_pct  gmm_Impulso_pct  gmm_Estrés_pct  prob_max_media  dudosas_pct
celda                                                                                                                                     
entrenamiento / entrenamiento       56.349              0.153         44.868           39.024          16.108           0.892        5.954
entrenamiento / validacion          59.474              0.145         58.839           32.563           8.598           0.896        